# 01 - Data collection

Loads the raw exoplanet, flare and observing-time catalogues from `D:\Research\data`, builds the master planet table, and writes the intermediate CSV files that notebook 02 consumes. Cell order is preserved from the original `habitability_pipeline.ipynb`; outputs are cleared.

In [ ]:
!pip install numpy pandas scikit-learn xgboost shap matplotlib

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER = r"D:\Research\data"
def P(name): return os.path.join(FOLDER, name)

# ---------- physics and label settings (FIXED, never edited after seeing results) ----------
D_REF      = 0.16            # AU, planet distance in Tilley et al. 2019
CME_P      = 0.47            # Lu et al. 2025, central value
CME_RANGE  = (0.35, 0.60)    # Lu et al. 2025, plausible range (used in Monte Carlo)
THR_HIGH   = 1.0             # R_eff thresholds (events per month at D_REF)
THR_MED    = 0.1
LOGE_BIG   = 34.0            # big flare = energy >= 1e34 erg

# ---------- observing time assumptions (months) ----------
DAYS_PER_MONTH     = 30.44
OBS_MONTHS_KEPLER  = 51.0                 # 17 quarters x 3 months (approximation, state in report)
OBS_MONTHS_GUNTHER = 2 * 27.4 / 30.44     # TESS sectors 1-2
OBS_MONTHS_YANG    = 1.8                  # assumed TESS observing time (limitation, state in report)

N_DRAW = 1000    # Monte Carlo draws per planet
SEED   = 42

In [ ]:
print(os.listdir(FOLDER))   # Windows adds extra .txt/.dat suffixes, check the exact names

F = dict(
    nasa   = "nasa_exoplanet_archive.csv.csv",
    howard = "howard2022_toi_flare_rates.csv.csv",
    koi    = "koi_cumulative.csv.csv",
    kep1   = "yangliu_kepler_table1.dat.txt",
    kep2   = "yangliu_kepler_table2.dat.dat",
    gun1   = "gunther2020_tess_table1.dat.dat",
    yang   = "table1.dat",            # Yang 2023 TESS
    stz2   = "table2.dat.txt",        # Stelzer 2022 flares
    stza3  = "tablea3.dat.txt",       # Stelzer 2022 observing times
)
missing = [v for v in F.values() if not os.path.exists(P(v))]
print("missing files:", missing)

In [ ]:
nasa = pd.read_csv(P(F["nasa"]), comment="#", low_memory=False)
nasa = nasa[nasa["default_flag"] == 1].copy()
nasa["TICID"] = pd.to_numeric(nasa["tic_id"].astype(str).str.replace("TIC ", "", regex=False), errors="coerce")
nasa_teff = nasa.dropna(subset=["TICID"]).drop_duplicates("TICID").set_index("TICID")["st_teff"]
print("NASA:", nasa.shape, "| with TIC:", nasa["TICID"].notna().sum())

In [ ]:
# MinUse_Erg = SMALLEST flare energy used for detection, not the largest flare.
howard = pd.read_csv(P(F["howard"]))
howard.columns = howard.columns.str.strip()
howard = howard.replace(-99, np.nan)
for c in ["orb_dist_au", "superflare_rate", "esuperflarerate", "Esuperflarerate", "oneminalpha_ffd", "beta_ffd"]:
    if c in howard.columns:
        howard[c] = howard[c].replace(-1, np.nan)
for c in ["TICID", "TOI", "N_flares", "obs_time_d", "orb_dist_au", "superflare_rate"]:
    howard[c] = pd.to_numeric(howard[c], errors="coerce")
print("Howard:", howard.shape, "| TOIs with flares:", (howard["N_flares"] > 0).sum())
print(list(howard.columns))

In [ ]:
k1 = pd.read_csv(P(F["kep1"]), sep="|", skiprows=3, header=None,
                 names=["KIC","Nf","Teff","logg","Lum","Prot","r_Prot","NS"], skipinitialspace=True)
k1["KIC"]  = pd.to_numeric(k1["KIC"],  errors="coerce")
k1["Teff"] = pd.to_numeric(k1["Teff"], errors="coerce")
k1 = k1.dropna(subset=["KIC"]).copy()
k1["KIC"] = k1["KIC"].astype("int64")

k2 = pd.read_csv(P(F["kep2"]), sep=r"\s+", header=None, names=["KIC","Quarter","tstart","tstop","logE"])
k2 = k2.apply(pd.to_numeric, errors="coerce").dropna(subset=["KIC","logE"])
k2["KIC"] = k2["KIC"].astype("int64")

koi = pd.read_csv(P(F["koi"]), comment="#")
print("Kepler stars:", len(k1), "| flares:", len(k2), "| KOI rows:", len(koi))

In [ ]:
# Gunther 2020 TESS (per-flare table). Duplicate-flagged flares are kept (note in report).
g1 = pd.read_fwf(P(F["gun1"]), header=None,
                 colspecs=[(0,9),(10,11),(99,108),(159,167),(168,173),(249,258)],
                 names=["TIC","Sec","Ebol","Tmag","Teff","Dup"])
for c in ["TIC","Ebol","Teff"]:
    g1[c] = pd.to_numeric(g1[c], errors="coerce")
print("Gunther flares:", len(g1), "(expect 8695) | flagged duplicates:", (pd.to_numeric(g1["Dup"], errors="coerce") > 0).sum())

# Yang 2023 TESS
y23 = pd.read_fwf(P(F["yang"]), colspecs=[(4,14),(72,78),(87,95)], names=["TIC","Teff","E"])
for c in ["TIC","Teff","E"]:
    y23[c] = pd.to_numeric(y23[c], errors="coerce")
print("Yang 2023 flares:", len(y23))

# Stelzer 2022: flares and observing time per star
s2 = pd.read_fwf(P(F["stz2"]),  colspecs=[(0,9),(54,59)], names=["TIC","logE"])
a3 = pd.read_fwf(P(F["stza3"]), colspecs=[(0,9),(34,40)], names=["TIC","obs_d"])
for d_, cols in ((s2, ["TIC","logE"]), (a3, ["TIC","obs_d"])):
    for c in cols:
        d_[c] = pd.to_numeric(d_[c], errors="coerce")
print("Stelzer flares:", len(s2), "| stars with obs time:", a3["obs_d"].notna().sum())

In [ ]:
def tic_key(s): return "TIC" + s.astype("int64").astype(str)

gg = g1.dropna(subset=["TIC","Ebol"]);  gg = gg[gg["Ebol"] > 0]
yy = y23.dropna(subset=["TIC","E"]);    yy = yy[yy["E"] > 0]
ss = s2.dropna(subset=["TIC","logE"])

flares = pd.concat([
    pd.DataFrame({"source":"Kepler",      "star_key":"KIC" + k2["KIC"].astype(str),  "logE":k2["logE"].astype(float)}),
    pd.DataFrame({"source":"Gunther2020", "star_key":tic_key(gg["TIC"]), "logE":np.log10(gg["Ebol"])}),
    pd.DataFrame({"source":"Yang2023",    "star_key":tic_key(yy["TIC"]), "logE":np.log10(yy["E"])}),
    pd.DataFrame({"source":"Stelzer2022", "star_key":tic_key(ss["TIC"]), "logE":ss["logE"].astype(float)}),
], ignore_index=True)

# star-level table: Teff (catalogue) and observing months
a3v = a3.dropna(subset=["TIC","obs_d"]);  a3v = a3v[a3v["obs_d"] > 0].drop_duplicates("TIC")
meta_g = gg.groupby("TIC", as_index=False).agg(Teff_cat=("Teff","first"))
meta_y = yy.groupby("TIC", as_index=False).agg(Teff_cat=("Teff","median"))

meta = pd.concat([
    pd.DataFrame({"source":"Kepler", "star_key":"KIC" + k1["KIC"].astype(str), "Teff_cat":k1["Teff"], "obs_months":OBS_MONTHS_KEPLER}),
    pd.DataFrame({"source":"Gunther2020", "star_key":tic_key(meta_g["TIC"]), "Teff_cat":meta_g["Teff_cat"], "obs_months":OBS_MONTHS_GUNTHER}),
    pd.DataFrame({"source":"Yang2023",    "star_key":tic_key(meta_y["TIC"]), "Teff_cat":meta_y["Teff_cat"], "obs_months":OBS_MONTHS_YANG}),
    pd.DataFrame({"source":"Stelzer2022", "star_key":tic_key(a3v["TIC"]),    "Teff_cat":np.nan, "obs_months":a3v["obs_d"] / DAYS_PER_MONTH}),
], ignore_index=True).drop_duplicates(["source","star_key"])

# Teff look-ups used to fill missing temperatures for TESS stars
yang_teff = meta_y.set_index("TIC")["Teff_cat"]
gun_teff  = meta_g.set_index("TIC")["Teff_cat"]

def resolve_teff(df):
    t = df["Teff_cat"].copy()
    is_tic = df["star_key"].str.startswith("TIC")
    tic = pd.to_numeric(df["star_key"].where(is_tic).str.replace("TIC", "", regex=False), errors="coerce")
    for mp in (nasa_teff, yang_teff, gun_teff):
        t = t.fillna(tic.map(mp))
    return t

print(flares.groupby("source")["logE"].describe()[["count","min","50%","max"]])

In [ ]:
def fit_alpha(logE, ec, min_n=20):
    """Maximum-likelihood power-law exponent alpha (dN/dE ~ E^-alpha) for flares with logE >= ec."""
    x = np.asarray(logE, float); x = x[x >= ec]
    n = len(x)
    if n < min_n: return np.nan, n
    s = np.log(10) * np.sum(x - ec)
    return float(np.clip(1 + n / s, 1.3, 3.0)), n

EC, ALPHA = {}, {}
for src, f in flares.groupby("source"):
    ec = min(float(np.quantile(f["logE"], 0.25)), LOGE_BIG)   # cutoff: never above 1e34
    a, n = fit_alpha(f["logE"], ec)
    EC[src], ALPHA[src] = ec, (a if np.isfinite(a) else 2.0)
    print(f"{src:12s} cutoff logE={ec:.2f}  alpha={ALPHA[src]:.2f}  (flares used: {n})")

In [ ]:
def star_table(src):
    f = flares[flares["source"] == src].copy()
    ec, alpha = EC[src], ALPHA[src]
    f["big"]   = f["logE"] >= LOGE_BIG
    f["above"] = f["logE"] >= ec
    g = f.groupby("star_key")
    st = pd.DataFrame({"n_all": g.size(), "max_logE": g["logE"].max(),
                       "n_big": g["big"].sum(), "n_above": g["above"].sum()})
    st = st.join(meta[meta["source"] == src].set_index("star_key")[["Teff_cat","obs_months"]], how="inner")
    st = st[st["obs_months"] > 0].reset_index()
    T = st["obs_months"].values
    factor = 10 ** ((LOGE_BIG - ec) * (1 - alpha))
    cnt = (st["n_big"] > 0).values

    st["rate_basis"]       = np.where(cnt, "count", "extrap")
    st["flare_rate_all"]   = st["n_all"] / T
    st["flare_rate_month"] = np.where(cnt, st["n_big"] / T, st["n_above"] / T * factor)
    st["gam_shape"]        = np.where(cnt, st["n_big"] + 0.5, st["n_above"] + 0.5)
    st["gam_scale"]        = np.where(cnt, 1.0 / T, factor / T)
    st["lam_sigma"]        = np.nan
    st["source"] = src
    return st

star_tabs = {s: star_table(s) for s in ["Kepler","Gunther2020","Yang2023","Stelzer2022"]}
for s, t in star_tabs.items():
    print(f"{s:12s} stars: {len(t):5d} | count-based: {(t.rate_basis=='count').sum():5d} | extrapolated: {(t.rate_basis=='extrap').sum():5d}")

In [ ]:
# superflare_rate = rate of flares >= 1e34 erg per YEAR (with 1-sigma errors); N_flares = flares detected above MinUse_Erg.
h = howard[(howard["N_flares"] > 0) & (howard["obs_time_d"] > 0) & (howard["orb_dist_au"] > 0)].dropna(subset=["TICID"]).copy()

lam = h["superflare_rate"] / 12.0                              # per month
if {"oneminalpha_ffd", "beta_ffd"} <= set(h.columns):          # fallback: fitted FFD at 1e34 erg (per day -> per month)
    lam = lam.fillna(10 ** (h["oneminalpha_ffd"] * LOGE_BIG + h["beta_ffd"]) * DAYS_PER_MONTH)

if {"esuperflarerate", "Esuperflarerate"} <= set(h.columns):
    sig = h[["esuperflarerate", "Esuperflarerate"]].mean(axis=1) / 12.0
else:
    sig = pd.Series(np.nan, index=h.index)
sig = sig.fillna(lam / np.sqrt(h["N_flares"]))                 # fallback: Poisson-like error

t_cols = [c for c in h.columns if "teff" in c.lower()]
h_teff = h[t_cols[0]] if t_cols else pd.Series(np.nan, index=h.index)

howard_rows = pd.DataFrame({
    "source": "Howard2022", "star_id": "TIC" + h["TICID"].astype("int64").astype(str),
    "planet": "TOI-" + h["TOI"].astype(str), "koi_sma": h["orb_dist_au"],
    "Teff_cat": h_teff, "n_all": h["N_flares"], "n_big": np.nan, "max_logE": np.nan,
    "obs_months": h["obs_time_d"] / DAYS_PER_MONTH,
    "flare_rate_all": h["N_flares"] / (h["obs_time_d"] / DAYS_PER_MONTH),
    "flare_rate_month": lam, "rate_basis": "howard_ffd",
    "gam_shape": np.nan, "gam_scale": np.nan, "lam_sigma": sig,
}).dropna(subset=["flare_rate_month"])
howard_rows["star_key"] = howard_rows["star_id"]
print("Howard rows with flares, distance and rate:", len(howard_rows))

In [ ]:
# ---------- Kepler planets ----------
koi_pl = koi[koi["koi_disposition"].isin(["CONFIRMED","CANDIDATE"]) & (koi["koi_sma"] > 0)].copy()
koi_pl["star_key"] = "KIC" + koi_pl["kepid"].astype("int64").astype(str)
kep = star_tabs["Kepler"].merge(
    koi_pl[["star_key","kepoi_name","koi_sma","koi_steff"]], on="star_key", how="inner")
kep["Teff_cat"] = kep["koi_steff"].fillna(kep["Teff_cat"])
kep = kep.rename(columns={"kepoi_name":"planet"}).drop(columns=["koi_steff"])
print("Kepler planets with flare stars:", len(kep))

# ---------- TESS planets (NASA confirmed + Howard-only candidates) ----------
nasa_pl = nasa[(nasa["pl_orbsmax"] > 0) & nasa["TICID"].notna()].copy()
nasa_pl = pd.DataFrame({"star_key": tic_key(nasa_pl["TICID"]), "planet": nasa_pl["pl_name"], "koi_sma": nasa_pl["pl_orbsmax"]})

h_only = howard[(howard["orb_dist_au"] > 0)].dropna(subset=["TICID"]).copy()
h_only = h_only[~h_only["TICID"].astype("int64").isin(nasa["TICID"].dropna().astype("int64"))]
h_only = pd.DataFrame({"star_key": tic_key(h_only["TICID"]), "planet": "TOI-" + h_only["TOI"].astype(str), "koi_sma": h_only["orb_dist_au"]})
tess_planets = pd.concat([nasa_pl, h_only], ignore_index=True).drop_duplicates(["star_key","planet"])

tess_parts = []
for s in ["Gunther2020","Yang2023","Stelzer2022"]:
    part = star_tabs[s].merge(tess_planets, on="star_key", how="inner")
    print(f"{s:12s} planets with flare stars: {len(part)}")
    tess_parts.append(part)

kep["star_id"] = kep["star_key"]
for p_ in tess_parts: p_["star_id"] = p_["star_key"]

In [ ]:
COMMON = ["source","star_id","planet","koi_sma","Teff_cat","star_key","n_all","n_big","max_logE","obs_months",
          "flare_rate_all","flare_rate_month","rate_basis","gam_shape","gam_scale","lam_sigma"]

allrows = pd.concat([kep[COMMON], howard_rows[COMMON]] + [p_[COMMON] for p_ in tess_parts], ignore_index=True)

# keep one source per star (highest priority)
PRIORITY = ["Kepler","Howard2022","Stelzer2022","Gunther2020","Yang2023"]
allrows["prio"] = allrows["source"].map({s: i for i, s in enumerate(PRIORITY)})
allrows = allrows[allrows["prio"] == allrows.groupby("star_id")["prio"].transform("min")]
allrows = allrows.drop_duplicates(["star_id","planet"]).drop(columns="prio").reset_index(drop=True)

# Teff -> star_type: F=0 (>6000 K), G=1 (5200-6000), K=2 (3700-5200), M=3 (<3700)
allrows["Teff"] = resolve_teff(allrows)
allrows = allrows.dropna(subset=["Teff","koi_sma","flare_rate_month"])
allrows = allrows[allrows["koi_sma"] > 0].copy()
allrows["star_type"] = np.select([allrows["Teff"] > 6000, allrows["Teff"] > 5200, allrows["Teff"] > 3700], [0, 1, 2], default=3)
allrows = allrows.drop(columns=["star_key","Teff_cat"])

print(len(allrows), "planet rows,", allrows["star_id"].nunique(), "unique stars")
print(allrows["source"].value_counts())
print(allrows["rate_basis"].value_counts())

In [ ]:
print([n for n in ["flares","meta","star_tabs","howard_rows","kep","tess_parts"] if n not in globals()])

In [ ]:
print(nasa[(nasa["pl_orbsmax"].isna()) & nasa["pl_orbper"].notna() & nasa["st_mass"].notna()].shape[0])

In [ ]:
# 1) check the formula against planets that have both values (median ratio should be close to 1)
nasa["a_kepler"] = (nasa["st_mass"] * (nasa["pl_orbper"] / 365.25) ** 2) ** (1 / 3)
both = nasa.dropna(subset=["pl_orbsmax", "a_kepler"])
print("median a_kepler / pl_orbsmax:", (both["a_kepler"] / both["pl_orbsmax"]).median().round(3))

# 2) how many of the 2,237 have a star with detected flares
need = nasa[nasa["pl_orbsmax"].isna() & nasa["a_kepler"].notna() & nasa["TICID"].notna()]
flare_keys = set().union(*[set(star_tabs[s]["star_key"]) for s in ["Gunther2020", "Yang2023", "Stelzer2022"]])
print("new planets with flare stars:", need[tic_key(need["TICID"]).isin(flare_keys)].shape[0])

In [ ]:
print(open(P("ReadMe.txt"), errors="ignore").read()[:3000])                 # catalogue description and column layout
print(open(P("table2.dat.txt"), errors="ignore").read()[:400])             # what is this table?
import json; j = json.load(open(P("A15.json"))); print(type(j), list(j)[:10] if isinstance(j, dict) else len(j))

In [ ]:
%pip install astropy

In [ ]:
import os
for name in ["ajad7956t9_mrt", "ajad7956t10_mrt"]:
    path = next((P(f) for f in os.listdir(FOLDER) if f.startswith(name)), None)
    print("=====", name, "->", path)
    print("".join(open(path, errors="ignore").readlines()[:45]))

In [ ]:
path10 = next(P(f) for f in os.listdir(FOLDER) if f.startswith("ajad7956t10_mrt"))
print("".join(open(path10, errors="ignore").readlines()[:60]))

In [ ]:
from io import StringIO
rows = [l for l in open(P("ajad7956t9_mrt.txt"), errors="ignore") if l[:10].strip().isdigit()]
t9 = pd.read_fwf(StringIO("".join(rows)), header=None,
                 colspecs=[(0,10),(46,55),(56,65),(80,89)], names=["TIC","Ef","Ef_cal","t_start"])
for c in t9.columns: t9[c] = pd.to_numeric(t9[c], errors="coerce")
print(t9.shape, "| flaring stars:", t9["TIC"].nunique())
print(t9[["Ef","Ef_cal"]].describe().round(2))

new  = set(t9["TIC"].dropna().astype("int64"))
host = {int(s[3:]) for s in tess_planets["star_key"]}              # TESS stars with a planet and a real distance
used = {int(s[3:]) for s in train["star_id"] if s.startswith("TIC")}
print("Lin stars that host a planet:", len(new & host), "| already in training:", len(new & used),
      "| NEW:", len((new & host) - used))

In [ ]:
import os
FOLDER = r"D:\Research\data"
def P(name): return os.path.join(FOLDER, name)

In [ ]:
import os, pandas as pd

files = sorted(f for f in os.listdir(FOLDER) if os.path.isfile(P(f)))

def first_line(f):
    with open(P(f), errors="ignore") as fh:
        for line in fh:
            if line.strip(): return line.strip()[:70]
    return ""

print(pd.DataFrame({"file": files,
                    "size_KB": [round(os.path.getsize(P(f)) / 1024, 1) for f in files],
                    "first line": [first_line(f) for f in files]}).to_string(index=False))

HAVE_KEYS = {
    "NASA Planetary Systems (default values)": ["nasa_exoplanet_archive"],
    "KOI cumulative (Kepler planets)":         ["koi_cumulative"],
    "Howard 2022 TOI flare rates":             ["howard2022"],
    "Kepler flares, Yang & Liu 2019":          ["yangliu_kepler"],
    "TESS flares, Gunther 2020":               ["gunther2020_tess_table1"],
    "TESS flares, Yang 2023 (table1.dat)":     ["table1.dat"],
    "Stelzer 2022 flares / obs time":          ["table2.dat", "tablea3"],
    "Lin 2024 flares (table 9)":               ["ajad7956t9"],
    "Lin 2024 flaring stars (table 10)":       ["ajad7956t10"],
}
TO_GET_KEYS = {
    "NASA Composite planet table":             ["pscomppars"],
    "TESS TOI candidates":                     ["toi_20"],
    "K2 planets and candidates":               ["k2pandc"],
    "Davenport 2016 Kepler flares":            ["davenport"],
    "A&A 689 A103 (TESS 20 s)":                ["a103"],
    "Doyle 2019 / 2020 TESS flares":           ["doyle"],
    "Raetz 2020 K2 M dwarfs":                  ["raetz"],
    "Ilin 2021 K2 clusters":                   ["ilin"],
}
low = [f.lower() for f in files]
def found(keys): return [files[i] for i, f in enumerate(low) if any(k in f for k in keys)]

print("\nHAVE")
for name, keys in HAVE_KEYS.items():
    print(f"  {'OK     ' if found(keys) else 'MISSING'}  {name:42s} {found(keys)}")
print("\nTO DOWNLOAD")
for name, keys in TO_GET_KEYS.items():
    print(f"  {'found  ' if found(keys) else 'not yet'}  {name:42s} {found(keys)}")

In [ ]:
def load(key):
    fn = next(f for f in os.listdir(FOLDER) if f.lower().startswith(key))
    return pd.read_csv(P(fn), comment="#", low_memory=False)

tables = {"PSCompPars": load("pscomppars"), "TOI": load("toi_20"), "K2": load("k2pandc")}
for name, d in tables.items():
    cols = [c for c in d.columns if any(k in c.lower() for k in ["tic", "epic", "tid", "orbsmax", "orbper", "st_mass", "st_teff", "default_flag", "disp"])]
    print("\n==", name, d.shape); print(cols)

In [ ]:
def tic_set(s): return set(pd.to_numeric(s.astype(str).str.replace("TIC ", "", regex=False), errors="coerce").dropna().astype("int64"))

comp, toi, k2 = tables["PSCompPars"], tables["TOI"], tables["K2"]
print("TOI columns present:", [c for c in ["st_logg", "st_rad", "st_mass", "pl_rade", "toi", "toi_name"] if c in toi.columns])

sets = {"PSCompPars (confirmed)": tic_set(comp["tic_id"]),
        "TOI (not FP/FA)":        tic_set(toi.loc[~toi["tfopwg_disp"].isin(["FP", "FA"]), "tid"]),
        "K2 (default rows)":      tic_set(k2.loc[k2["default_flag"] == 1, "tic_id"])}
for name, s in sets.items():
    print(f"{name:24s} hosts with TIC: {len(s):5d} | also Lin flaring stars: {len(s & new):4d}")
allp = set().union(*sets.values())
print("all three combined       hosts:", len(allp), "| Lin flaring stars that host a planet:", len(allp & new))
print("planets with a catalogued distance:", int(comp["pl_orbsmax"].notna().sum()), "(PSCompPars),",
      int(k2.loc[k2["default_flag"] == 1, "pl_orbsmax"].notna().sum()), "(K2 default rows)")

In [ ]:
from io import StringIO
rows = [l for l in open(P("ajad7956t9_mrt.txt"), errors="ignore") if l[:10].strip().isdigit()]
t9 = pd.read_fwf(StringIO("".join(rows)), header=None,
                 colspecs=[(0,10),(46,55),(56,65),(80,89)], names=["TIC","Ef","Ef_cal","t_start"])
for c in t9.columns: t9[c] = pd.to_numeric(t9[c], errors="coerce")
new = set(t9["TIC"].dropna().astype("int64"))          # TIC IDs of all Lin flaring stars
print(len(t9), "flares |", len(new), "flaring stars")  # expect about 249,562 and 18,032

In [ ]:
%pip install astroquery
from astroquery.vizier import Vizier

CATS = {"davenport": "J/ApJ/829/23",       # Kepler flares, Davenport 2016
        "a103":      "J/A+A/689/A103",     # TESS 20-second cadence
        "doyle2020": "J/MNRAS/494/3596",
        "doyle2019": "J/MNRAS/489/437",
        "raetz":     "J/A+A/637/A22",      # K2 M dwarfs
        "ilin":      "J/A+A/645/A116"}     # K2 open clusters (this ID is unchecked, see below)

for key, cat in CATS.items():
    try:
        res = Vizier(columns=["**"], row_limit=-1).get_catalogs(cat)
        for t in res:
            name = t.meta.get("name", "table").replace("/", "_")
            t.to_pandas().to_csv(P(f"{key}_{name}.csv"), index=False)
            print(f"{key:10s} {name:32s} rows: {len(t):8d}  columns: {t.colnames[:8]}")
    except Exception as e:
        print("FAILED", key, cat, "->", e)

In [ ]:
import glob
def ids(s): return set(pd.to_numeric(s.astype(str).str.replace("TIC", "").str.replace("EPIC", "").str.strip(), errors="coerce").dropna().astype("int64"))

# 1) what did each download contain? (per-flare tables repeat the same star id on many rows)
for key in ["davenport", "a103", "doyle2020", "doyle2019", "raetz"]:
    for f in sorted(glob.glob(P(f"{key}_*.csv"))):
        d = pd.read_csv(f)
        idc = next((c for c in d.columns if c.upper() in ("TIC", "KIC", "EPIC", "ID")), None)
        print(f"\n{os.path.basename(f)}  rows={len(d)}  id column={idc}  unique ids={d[idc].nunique() if idc else None}")
        print(d.columns.tolist())

# 2) how many planet hosts does each catalogue reach?
koi_ = pd.read_csv(P(F["koi"]), comment="#", low_memory=False)
koi_hosts = ids(koi_.loc[koi_["koi_disposition"].isin(["CONFIRMED", "CANDIDATE"]) & (koi_["koi_sma"] > 0), "kepid"])
k2h = ids(tables["K2"].loc[tables["K2"]["default_flag"] == 1, "epic_hostname"])
planet_tics = set(mp["tic"])
def rd(key): return pd.read_csv(sorted(glob.glob(P(f"{key}_*.csv")))[0])
dav, a103, d20, rz = rd("davenport"), rd("a103"), rd("doyle2020"), rd("raetz")

print("\nDavenport KIC stars that host a KOI planet with a distance:", len(ids(dav["KIC"]) & koi_hosts))
print("A&A 689 A103 TIC stars that host a planet:                  ", len(ids(a103["TIC"]) & planet_tics))
print("Doyle 2020 TIC stars that host a planet:                     ", len(ids(d20["TIC"]) & planet_tics))
print("Raetz 2020 EPIC stars that host a K2 planet:                 ", len(ids(rz["EPIC"]) & k2h))

In [ ]:
for cat, cols in {"J/ApJ/829/23": ["Nfl", "Nfl68", "alpha", "beta"],
                  "J/A+A/637/A22": ["Nflares(SC)", "nu(SC)", "logE(Fmax)"],
                  "J/MNRAS/494/3596": ["Nf", "Sector", "logEmin", "logEmax", "tfmin", "tfmax"],
                  "J/A+A/689/A103": ["Ebol", "TimeS"]}.items():
    t = Vizier(columns=["**"], row_limit=1).get_catalogs(cat)[0]
    print("\n==", cat)
    for c in cols:
        if c in t.colnames: print(f"  {c:12s} unit={t[c].unit}  {t[c].description}")

In [ ]:
import os, glob
import numpy as np, pandas as pd

FOLDER = r"D:\Research\data"
def P(name): return os.path.join(FOLDER, name)
def ids(s): return set(pd.to_numeric(s.astype(str).str.replace("TIC", "").str.replace("EPIC", "").str.strip(), errors="coerce").dropna().astype("int64"))
def load(key):
    fn = next(f for f in os.listdir(FOLDER) if f.lower().startswith(key))
    return pd.read_csv(P(fn), comment="#", low_memory=False)

# planet hosts with a TIC ID (confirmed + TOI candidates + K2), as in the earlier overlap count
comp, toi, k2 = load("pscomppars"), load("toi_20"), load("k2pandc")
planet_tics = (ids(comp["tic_id"])
               | ids(toi.loc[~toi["tfopwg_disp"].isin(["FP", "FA"]), "tid"])
               | ids(k2.loc[k2["default_flag"] == 1, "tic_id"]))

# Lin 2024 flaring stars (TIC is in the first 10 characters of each data line)
lines = [l for l in open(P("ajad7956t9_mrt.txt"), errors="ignore") if l[:10].strip().isdigit()]
new = set(pd.to_numeric(pd.Series([l[:10] for l in lines]), errors="coerce").dropna().astype("int64"))

# A&A 689 A103
a103 = pd.read_csv(sorted(glob.glob(P("a103_*.csv")))[0])
print(a103["Ebol"].describe())
a_tics = ids(a103["TIC"])
print("\nplanet hosts (all tables):", len(planet_tics))
print("Lin flaring stars:", len(new), "| that host a planet:", len(new & planet_tics))
print("A103 flaring stars:", len(a_tics), "| that host a planet:", len(a_tics & planet_tics))
print("A103 planet hosts not in Lin:", len((a_tics & planet_tics) - new))

In [ ]:
e = pd.to_numeric(a103["Ebol"], errors="coerce")
print("no energy:", e.isna().sum(), "| with energy:", e.notna().sum())
print(e.describe(percentiles=[.05, .5, .95]))
print(a103.loc[e.isna(), "Ebol"].value_counts().head())

In [ ]:
from astroquery.mast import Observations
tic = sorted(a_tics & planet_tics)[0]
o = Observations.query_criteria(obs_collection="TESS", dataproduct_type="timeseries", target_name=str(tic)).to_pandas()
cols = [c for c in ["obs_id", "sequence_number", "t_exptime", "t_min", "t_max", "provenance_name"] if c in o.columns]
print(tic, len(o), "observations"); print(o[cols].head(12)); print(o.columns.tolist())

In [ ]:
from astroquery.mast import Observations
import time

# time range covered by each catalogue's own flare list, in MJD
def to_mjd(x): return x + 56999.5 if x.median() < 50000 else x            # BTJD -> MJD, if the column is BTJD
lin_t = to_mjd(pd.to_numeric(pd.Series([l[80:89] for l in lines]), errors="coerce").dropna())
a_t   = to_mjd(pd.to_numeric(a103["TimeS"], errors="coerce").dropna())
print("Lin  flare times (MJD):", round(lin_t.min()), "to", round(lin_t.max()))
print("A103 flare times (MJD):", round(a_t.min()),  "to", round(a_t.max()))

def sectors(tic):                                                           # single-sector SPOC light curves of one star
    o = Observations.query_criteria(obs_collection="TESS", dataproduct_type="timeseries", target_name=str(tic)).to_pandas()
    o = o[o["provenance_name"] == "SPOC"]
    o = o[o["obs_id"].str.findall(r"-s(\d{4})").str.len() == 1]               # drop multi-sector products
    return o[["sequence_number", "t_exptime", "t_min", "t_max"]]

def months(o, cad_lo, cad_hi, t0, t1, max_sector=None):                    # observing months for one catalogue's cadence and date range
    o = o[o["t_exptime"].between(cad_lo, cad_hi) & (o["t_min"] >= t0 - 30) & (o["t_max"] <= t1 + 30)]
    if max_sector: o = o[o["sequence_number"] <= max_sector]
    o = o.drop_duplicates("sequence_number")
    return len(o), (o["t_max"] - o["t_min"]).sum() / 30.44

todo = sorted((new | a_tics) & planet_tics)
out_file, rows = P("mast_obs_times.csv"), []
done = set(pd.read_csv(out_file)["TIC"]) if os.path.exists(out_file) else set()
if done: rows = pd.read_csv(out_file).to_dict("records")

for i, tic in enumerate(todo):
    if tic in done: continue
    try:
        o = sectors(tic)
        nl, ml = months(o, 100, 140, lin_t.min(), lin_t.max(), max_sector=72)
        na, ma = months(o, 15, 25,  a_t.min(),  a_t.max())
        rows.append({"TIC": tic, "lin_sectors": nl, "lin_months": ml, "a103_sectors": na, "a103_months": ma})
    except Exception as e:
        rows.append({"TIC": tic, "lin_sectors": np.nan, "lin_months": np.nan, "a103_sectors": np.nan, "a103_months": np.nan})
    if i % 25 == 0:
        pd.DataFrame(rows).to_csv(out_file, index=False); print(i, "/", len(todo), "done"); time.sleep(1)
obs = pd.DataFrame(rows); obs.to_csv(out_file, index=False)
print(obs.describe().round(2))
print("stars with no observing time found:", int(obs["lin_months"].isna().sum()))

In [ ]:
import os, glob
import numpy as np, pandas as pd

FOLDER = r"D:\Research\data"
def P(n): return os.path.join(FOLDER, n)
def load(key):
    fn = next(f for f in os.listdir(FOLDER) if f.lower().startswith(key))
    return pd.read_csv(P(fn), comment="#", low_memory=False)

comp, toi, k2 = load("pscomppars"), load("toi_20"), load("k2pandc")

def to_tic(s): return pd.to_numeric(s.astype(str).str.replace("TIC", "", regex=False).str.strip(), errors="coerce")
def kepler3(mass, per_d): return (mass * (per_d / 365.25) ** 2) ** (1 / 3)      # AU, from measured period and stellar mass

a = pd.DataFrame({"planet": comp["pl_name"], "tic": to_tic(comp["tic_id"]), "per": comp["pl_orbper"],
                  "sma": comp["pl_orbsmax"], "mass": comp["st_mass"], "status": "CONFIRMED", "table": "PSCompPars"})
k = k2[k2["default_flag"] == 1]
k = k[~k["disposition"].astype(str).str.upper().str.contains("FALSE")]
b = pd.DataFrame({"planet": k["epic_candname"], "tic": to_tic(k["tic_id"]), "per": k["pl_orbper"],
                  "sma": k["pl_orbsmax"], "mass": k["st_mass"], "status": k["disposition"], "table": "K2"})
t = toi[~toi["tfopwg_disp"].isin(["FP", "FA"])]
c = pd.DataFrame({"planet": "TOI-" + t["toi"].astype(str), "tic": pd.to_numeric(t["tid"], errors="coerce"), "per": t["pl_orbper"],
                  "sma": np.nan, "mass": 10 ** (t["st_logg"] - 4.438) * t["st_rad"] ** 2,   # mass from catalogued logg and radius
                  "status": t["tfopwg_disp"], "table": "TOI"})

def drop_dups(low, high):                      # same star and orbital period within 2% = same planet
    m = low.reset_index().merge(high[["tic", "per"]], on="tic", suffixes=("", "_h"))
    m = m[(m["per"] / m["per_h"] - 1).abs() < 0.02]
    return low[~low.index.isin(m["index"])]

b = drop_dups(b.dropna(subset=["tic"]), a.dropna(subset=["tic"]))
c = drop_dups(c.dropna(subset=["tic"]), pd.concat([a, b]).dropna(subset=["tic"]))
mp = pd.concat([a, b, c], ignore_index=True).dropna(subset=["tic"])
mp["tic"] = mp["tic"].astype("int64")

mp["dist_basis"] = np.where(mp["sma"] > 0, "catalogued", np.where(mp["table"] == "TOI", "kepler3_logg_radius", "kepler3_period_mass"))
mp["koi_sma"] = mp["sma"].where(mp["sma"] > 0, kepler3(mp["mass"], mp["per"]))
mp = mp[mp["koi_sma"] > 0].drop(columns=["sma", "mass"]).reset_index(drop=True)
mp["star_key"] = "TIC" + mp["tic"].astype(str)

print("master planets with a distance:", len(mp), "| hosts:", mp["tic"].nunique())
print(mp.groupby(["table", "dist_basis"]).size())

In [ ]:
from astroquery.vizier import Vizier
cats = Vizier.find_catalogs("Pietras")
for k, v in cats.items():
    print(k, "|", v.description)

In [ ]:
import os, glob
D = r"D:\Research\data"
c = [f for f in glob.glob(rf"{D}\**\datafile*", recursive=True) if os.path.isfile(f)]
print(c)
f = c[0]
print("size KB:", os.path.getsize(f)//1024)
with open(f, "r", errors="replace") as fh:
    lines = fh.readlines()
print("total lines:", len(lines))
for i, l in enumerate(lines[:25]):
    print(i, repr(l[:200]))
print("...")
for l in lines[-3:]:
    print(repr(l[:200]))

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"

rows = []
with open(rf"{D}\datafile.txt", errors="replace") as fh:
    for l in fh:
        if l[:10].strip().isdigit() and len(l) > 55:
            rows.append(l.rstrip("\n"))
def num(s):
    s = s.strip()
    return float(s) if s else np.nan
p = pd.DataFrame({
    "tic": [int(l[0:10]) for l in rows],
    "sec": [int(l[11:13]) for l in rows],
    "E1":  [num(l[34:42]) for l in rows],
    "E2":  [num(l[43:51]) for l in rows],
    "pdf": [l[55:56] for l in rows]})
print("flares:", len(p), "| stars:", p.tic.nunique(), "| sectors:", p.sec.min(), "-", p.sec.max())
print("possible false (Y):", int((p.pdf == "Y").sum()))
for c in ["E1", "E2"]:
    v = np.log10(p[c][p[c] > 0])
    print(c, "null:", int(p[c].isna().sum()), "| log10 min/median/max:",
          round(v.min(), 2), round(v.median(), 2), round(v.max(), 2),
          "| flares >=1e34:", int((p[c] >= 1e34).sum()))

q = p[p.pdf != "Y"]
s = q.groupby("tic").agg(n_all=("tic", "size"),
                         n_energy=("E1", lambda x: int(x.notna().sum())),
                         n_big=("E1", lambda x: int((x >= 1e34).sum()))).reset_index()

def digits(ser):
    return set(pd.to_numeric(ser.astype(str).str.extract(r"(\d+)")[0], errors="coerce").dropna().astype(int))
hosts = set()
for fn, c in [("PSCompPars_2026.10.16_13.03.27.csv", "tic_id"),
              ("TOI_2026.10.16_13.07.57.csv", "tid"),
              ("k2pandc_2026.10.16_13.09.00.csv", "tic_id")]:
    t = pd.read_csv(rf"{D}\{fn}", comment="#", low_memory=False)
    if c in t.columns: hosts |= digits(t[c])
    else: print("missing column", c, "in", fn, list(t.columns)[:12])

v9 = pd.read_csv(rf"{D}\training_v9_split.csv")
have = digits(v9.star_id[v9.star_id.astype(str).str.startswith("TIC")])
s["host"] = s.tic.isin(hosts); s["in_v9"] = s.tic.isin(have)
new = s[s.host & ~s.in_v9]
print({"flaring_stars": len(s), "planet_hosts": int(s.host.sum()),
       "already_in_v9": int((s.host & s.in_v9).sum()),
       "NEW_hosts": len(new),
       "NEW_hosts_with_big_flare": int((new.n_big > 0).sum()),
       "NEW_hosts_no_energy_at_all": int((new.n_energy == 0).sum())})

In [ ]:
cand = new[new.n_energy > 0].copy()
cand["energy_partial"] = cand.n_energy < cand.n_all
cand[["tic", "n_all", "n_energy", "n_big", "energy_partial"]].to_csv(
    rf"{D}\pietras_new_hosts.csv", index=False)
print("hosts to fetch:", len(cand), "| partial energy:", int(cand.energy_partial.sum()))

In [ ]:
import pandas as pd, numpy as np, os, time
from astroquery.mast import Observations
D = r"D:\Research\data"
hosts = pd.read_csv(rf"{D}\pietras_new_hosts.csv")
out_f = rf"{D}\mast_obs_times_pietras.csv"

done = set()
if os.path.exists(out_f):
    o = pd.read_csv(out_f)
    done = set(o[o.err.isna()].tic)          # failed stars are retried
todo = [t for t in hosts.tic if t not in done]
print("to fetch:", len(todo), "| already done:", len(done))

def fetch(tic):
    obs = Observations.query_criteria(obs_collection="TESS",
          dataproduct_type="timeseries", provenance_name="SPOC",
          target_name=str(tic))
    if len(obs) == 0:
        return 0, 0.0, ""
    df = obs.to_pandas()
    df["sector"] = df.obs_id.str.extract(r"-s(\d{4})-")[0].astype(float)
    df = df[df.sector.notna() & (df.sector <= 39) & df.t_exptime.between(100, 140)]
    df = df.drop_duplicates("sector")        # each sector counted once
    days = float((df.t_max - df.t_min).sum())
    return df.sector.nunique(), days, ",".join(str(int(s)) for s in sorted(df.sector))

for i, t in enumerate(todo):
    try:
        ns, days, secs = fetch(t); err = np.nan
    except Exception as e:
        ns, days, secs, err = np.nan, np.nan, "", str(e)[:60]
    row = dict(tic=t, n_sectors=ns, obs_days=days,
               obs_months=days / 30.4375, sectors=secs, err=err)
    pd.DataFrame([row]).to_csv(out_f, mode="a", header=not os.path.exists(out_f), index=False)
    if i % 25 == 0:
        print(i, t, ns, round(days, 1) if days == days else days)
    time.sleep(0.3)
print("finished")

In [ ]:
   m = pd.read_csv(out_f)
   print(m.obs_months.describe().round(2).to_string())
   print({"stars": len(m), "no_time": int((m.n_sectors.fillna(0) == 0).sum()),
          "errors": int(m.err.notna().sum())})

In [ ]:
import pandas as pd, numpy as np, os, time
from astroquery.mast import Observations
D = r"D:\Research\data"
out_f = rf"{D}\mast_obs_times_pietras.csv"
if os.path.exists(out_f): os.remove(out_f)
hosts = pd.read_csv(rf"{D}\pietras_new_hosts.csv")

def fetch(tic):
    obs = Observations.query_criteria(obs_collection="TESS",
          dataproduct_type="timeseries", provenance_name="SPOC",
          target_name=str(tic))
    if len(obs) == 0:
        return 0, 0.0, ""
    df = obs.to_pandas()
    df = df[df.obs_id.str.startswith("tess") & df.obs_id.str.endswith("-s")]
    df["sector"] = df.obs_id.str.extract(r"-s(\d{4})-")[0].astype(float)
    df = df[df.sector.notna() & (df.sector <= 39)
            & df.t_exptime.between(100, 140)
            & ((df.t_max - df.t_min) <= 35)]
    df = df.drop_duplicates("sector")
    days = float((df.t_max - df.t_min).sum())
    return df.sector.nunique(), days, ",".join(str(int(s)) for s in sorted(df.sector))

for i, t in enumerate(hosts.tic):
    try:
        ns, days, secs = fetch(t); err = np.nan
    except Exception as e:
        ns, days, secs, err = np.nan, np.nan, "", str(e)[:60]
    pd.DataFrame([dict(tic=t, n_sectors=ns, obs_days=days, obs_months=days/30.4375,
                       sectors=secs, err=err)]).to_csv(out_f, mode="a",
                       header=not os.path.exists(out_f), index=False)
    time.sleep(0.3)

m = pd.read_csv(out_f)
m["days_per_sector"] = m.obs_days / m.n_sectors
print(m.days_per_sector.describe().round(1).to_string())
print("over 35 days per sector:", int((m.days_per_sector > 35).sum()))
print("under 15 days per sector:", int((m.days_per_sector < 15).sum()))
print({"stars": len(m), "no_time": int((m.n_sectors.fillna(0) == 0).sum()),
       "errors": int(m.err.notna().sum())})

In [ ]:
   m0 = pd.read_csv(rf"{D}\mast_obs_times.csv")
   print(list(m0.columns))
   if {"n_sectors", "obs_days"} <= set(m0.columns):
       r = m0.obs_days / m0.n_sectors
       print(r.describe().round(1).to_string())
       print("over 35 days per sector:", int((r > 35).sum()))
   else:
       print(m0.describe().round(2).to_string())

In [ ]:
from astroquery.vizier import Vizier
import os
Vizier.ROW_LIMIT = -1
t = Vizier.get_catalogs("J/ApJS/261/26")
for k in t.keys():
    print(k, len(t[k]), list(t[k].colnames)[:14])
    t[k].to_pandas().to_csv(rf"D:\Research\data\zhang_apjs261_26_{k.split('/')[-1]}.csv", index=False)

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
def ints(s):
    return set(pd.to_numeric(pd.Series(s).astype(str).str.extract(r"(\d+)")[0],
               errors="coerce").dropna().astype(int))

v = pd.read_csv(rf"{D}\training_v10_detected.csv")
sid = v.star_id.astype(str)
have = {"TIC": ints(sid[sid.str.startswith("TIC")]), "KIC": ints(sid[sid.str.startswith("KIC")])}

ps  = pd.read_csv(rf"{D}\PSCompPars_2026.10.16_13.03.27.csv", comment="#", low_memory=False)
toi = pd.read_csv(rf"{D}\TOI_2026.10.16_13.07.57.csv", comment="#", low_memory=False)
k2  = pd.read_csv(rf"{D}\k2pandc_2026.10.16_13.09.00.csv", comment="#", low_memory=False)
koi = pd.read_csv(rf"{D}\koi_cumulative.csv.csv", comment="#", low_memory=False)
if "default_flag" in k2.columns: k2 = k2[k2.default_flag == 1]
print("k2 id columns:", [c for c in k2.columns if "epic" in c.lower() or c == "tic_id"])
print("koi id columns:", [c for c in koi.columns if c in ("kepid", "kepoi_name", "koi_sma")])

planet_hosts = {"TIC": ints(ps.tic_id) | ints(toi.tid) | ints(k2.tic_id),
                "KIC": ints(koi.kepid) if "kepid" in koi.columns else set()}
ecol = next((c for c in k2.columns if "epic" in c.lower() and "host" in c.lower()), None)
planet_hosts["EPIC"] = ints(k2[ecol]) if ecol else set()
epic2tic = {}
if ecol and "tic_id" in k2.columns:
    for e, t in zip(pd.to_numeric(k2[ecol].astype(str).str.extract(r"(\d+)")[0], errors="coerce"),
                    pd.to_numeric(k2.tic_id.astype(str).str.extract(r"(\d+)")[0], errors="coerce")):
        if pd.notna(e) and pd.notna(t): epic2tic[int(e)] = int(t)

for tab, idc in [("table4", "TIC"), ("table5", "KIC"), ("table6", "EPIC")]:
    x = pd.read_csv(rf"{D}\zhang_apjs261_26_{tab}.csv")
    x["id"] = pd.to_numeric(x[idc], errors="coerce")
    x["E"] = pd.to_numeric(x["Energy"], errors="coerce")
    peak = pd.to_numeric(x["Peak"], errors="coerce")
    unit_erg = x.E.median() > 1e20
    big = (x.E >= 1e34) if unit_erg else (x.E >= 34)
    s = x.assign(big=big).groupby("id").agg(n=("id", "size"), nbig=("big", "sum")).reset_index()
    ids = set(s.id.dropna().astype(int))
    inv10 = {i for i in ids if (epic2tic.get(i) in have["TIC"] if idc == "EPIC" else i in have[idc])}
    matched = ids & planet_hosts[idc]
    new = (matched - inv10)
    print({"table": tab, "flares": len(x), "stars": len(ids),
           "E_raw_min/median/max": (float(x.E.min()), float(x.E.median()), float(x.E.max())),
           "looks_like_erg": bool(unit_erg), "flares>=1e34": int(big.sum()),
           "peak_min/max": (float(peak.min()), float(peak.max())),
           "stars_matched_to_planet_tables": len(matched), "already_in_v10": len(inv10),
           "NEW_hosts": len(new),
           "NEW_with_big_flare": int(s[s.id.isin(new)].nbig.gt(0).sum())})

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
def ints(s):
    return set(pd.to_numeric(pd.Series(s).astype(str).str.extract(r"(\d+)")[0],
               errors="coerce").dropna().astype(int))

v = pd.read_csv(rf"{D}\training_v10_detected.csv")
sid = v.star_id.astype(str)
have = {"TIC": ints(sid[sid.str.startswith("TIC")]), "KIC": ints(sid[sid.str.startswith("KIC")])}

ps  = pd.read_csv(rf"{D}\PSCompPars_2026.10.16_13.03.27.csv", comment="#", low_memory=False)
toi = pd.read_csv(rf"{D}\TOI_2026.10.16_13.07.57.csv", comment="#", low_memory=False)
k2  = pd.read_csv(rf"{D}\k2pandc_2026.10.16_13.09.00.csv", comment="#", low_memory=False)
koi = pd.read_csv(rf"{D}\koi_cumulative.csv.csv", comment="#", low_memory=False)
if "default_flag" in k2.columns: k2 = k2[k2.default_flag == 1]
print("k2 id columns:", [c for c in k2.columns if "epic" in c.lower() or c == "tic_id"])
print("koi id columns:", [c for c in koi.columns if c in ("kepid", "kepoi_name", "koi_sma")])

planet_hosts = {"TIC": ints(ps.tic_id) | ints(toi.tid) | ints(k2.tic_id),
                "KIC": ints(koi.kepid) if "kepid" in koi.columns else set()}
ecol = next((c for c in k2.columns if "epic" in c.lower() and "host" in c.lower()), None)
planet_hosts["EPIC"] = ints(k2[ecol]) if ecol else set()
epic2tic = {}
if ecol and "tic_id" in k2.columns:
    for e, t in zip(pd.to_numeric(k2[ecol].astype(str).str.extract(r"(\d+)")[0], errors="coerce"),
                    pd.to_numeric(k2.tic_id.astype(str).str.extract(r"(\d+)")[0], errors="coerce")):
        if pd.notna(e) and pd.notna(t): epic2tic[int(e)] = int(t)

for tab, idc in [("table4", "TIC"), ("table5", "KIC"), ("table6", "EPIC")]:
    x = pd.read_csv(rf"{D}\zhang_apjs261_26_{tab}.csv")
    x["id"] = pd.to_numeric(x[idc], errors="coerce")
    x["E"] = pd.to_numeric(x["Energy"], errors="coerce")
    peak = pd.to_numeric(x["Peak"], errors="coerce")
    unit_erg = x.E.median() > 1e20
    big = (x.E >= 1e34) if unit_erg else (x.E >= 34)
    s = x.assign(big=big).groupby("id").agg(n=("id", "size"), nbig=("big", "sum")).reset_index()
    ids = set(s.id.dropna().astype(int))
    inv10 = {i for i in ids if (epic2tic.get(i) in have["TIC"] if idc == "EPIC" else i in have[idc])}
    matched = ids & planet_hosts[idc]
    new = (matched - inv10)
    print({"table": tab, "flares": len(x), "stars": len(ids),
           "E_raw_min/median/max": (float(x.E.min()), float(x.E.median()), float(x.E.max())),
           "looks_like_erg": bool(unit_erg), "flares>=1e34": int(big.sum()),
           "peak_min/max": (float(peak.min()), float(peak.max())),
           "stars_matched_to_planet_tables": len(matched), "already_in_v10": len(inv10),
           "NEW_hosts": len(new),
           "NEW_with_big_flare": int(s[s.id.isin(new)].nbig.gt(0).sum())})

In [ ]:
import pandas as pd, numpy as np
D = r"D:\Research\data"
t = {k: pd.read_csv(rf"{D}\zhang_apjs261_26_table{n}.csv")
     for k, n in [("TESS", 4), ("Kepler", 5), ("K2", 6), ("T8", 8)]}
t8 = t["T8"].copy()
print("table8 columns:", list(t8.columns))
print(t8.head(2).to_string()[:700])
s = t8["ID"].astype(str)
t8["pref"] = s.str.extract(r"^([A-Za-z]+)")[0].fillna("TIC")
t8["num"] = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
t8["Sum"] = pd.to_numeric(t8["Sum"], errors="coerce")
t8["Nfl"] = pd.to_numeric(t8["Nfl"], errors="coerce")
print("table8 id prefixes:", t8.pref.value_counts().to_dict())

for name, idc in [("TESS", "TIC"), ("Kepler", "KIC"), ("K2", "EPIC")]:
    x = t[name].copy()
    x["id"] = pd.to_numeric(x[idc], errors="coerce")
    x["pk"] = pd.to_numeric(x["Peak"], errors="coerce")
    x["E"] = pd.to_numeric(x["Energy"], errors="coerce")
    x["R"] = pd.to_numeric(x["Rad"], errors="coerce")
    x = x.sort_values(["id", "pk"]).reset_index(drop=True)
    dup = (x.id == x.id.shift()) & ((x.pk - x.pk.shift()).abs() < 0.01)
    y = x[~dup].sort_values("pk").reset_index(drop=True)
    tol = 0.003
    close = y.pk.diff().abs() < tol
    cross = close & (y.id != y.id.shift())
    span = y.pk.max() - y.pk.min()
    expected = len(y) ** 2 * tol / span
    s8 = t8[t8.pref == idc]
    g = s8.groupby("num").Sum.agg(["min", "max"])
    ids = set(x.id.dropna().astype(int))
    n_tab = x.groupby("id").size().rename("n_tab")
    n8 = s8.groupby("num").Nfl.max().rename("n8")
    cmp = pd.concat([n_tab, n8], axis=1).dropna()
    big = y[(y.R <= 2) & (y.E >= 1e34)]
    print({"mission": name, "flares": len(x), "duplicate_flares": int(dup.sum()),
           "cross_star_pairs": int(cross.sum()), "expected_by_chance": round(expected, 1),
           "stars": len(ids), "stars_in_table8": len(ids & set(g.index.astype(int))),
           "table8_Sum_min/median/max": (round(s8.Sum.min(), 1), round(s8.Sum.median(), 1), round(s8.Sum.max(), 1)),
           "stars_Sum_differs_by_planet": int(((g["max"] - g["min"]).abs() > 0.01).sum()),
           "flare_count_matches_table8": f"{int((cmp.n_tab == cmp.n8).sum())}/{len(cmp)}",
           "flares_with_Rad>2": int((x.R > 2).sum()), "stars_with_Rad>2": int(x[x.R > 2].id.nunique()),
           "stars_with_flare>=1e34_and_Rad<=2": int(big.id.nunique())})

In [ ]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
D = r"D:\Research\data"
T = {k: pd.read_csv(rf"{D}\zhang_apjs261_26_table{n}.csv")
     for k, n in [("TESS", 4), ("Kepler", 5), ("K2", 6), ("T8", 8)]}
def num(s):
    return pd.to_numeric(pd.Series(s).astype(str).str.extract(r"(\d+)")[0], errors="coerce")
def col(df, c):
    return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)

v = pd.read_csv(rf"{D}\training_v10_detected.csv")
have_ids = set(v.star_id.astype(str).str.strip())
have_names = set(v.planet.astype(str).str.strip())

# observing time per star (table8)
t8 = T["T8"].copy()
t8["pref"] = t8.ID.astype(str).str.extract(r"^([A-Za-z]+)")[0]
t8["id"] = num(t8.ID)
t8["Sum"] = pd.to_numeric(t8.Sum, errors="coerce")
ts = t8.dropna(subset=["id", "Sum"]).drop_duplicates(["pref", "id"])[["pref", "id", "Sum"]]
ts["id"] = ts.id.astype("int64")

stages = {}
def stars_from(name, idc, pref):
    x = T[name].copy()
    x["id"] = pd.to_numeric(x[idc], errors="coerce")
    x["pk"] = pd.to_numeric(x["Peak"], errors="coerce")
    x["E"] = pd.to_numeric(x["Energy"], errors="coerce")
    x["R"] = pd.to_numeric(x["Rad"], errors="coerce")
    x["Teff"] = pd.to_numeric(x["Teff"], errors="coerce")
    x = x.dropna(subset=["id", "pk", "E"]).sort_values(["id", "pk"]).reset_index(drop=True)
    n_raw = len(x)
    dup = (x.id == x.id.shift()) & ((x.pk - x.pk.shift()).abs() < 0.01)
    x = x[~dup].sort_values("pk").reset_index(drop=True)
    n_dedup = len(x)
    tol = 0.003
    pair = (x.pk.diff().abs() < tol) & (x.id != x.id.shift())
    expected = len(x) ** 2 * tol / (x.pk.max() - x.pk.min())
    if pair.sum() > 3 * expected:
        x = x[~(pair | pair.shift(-1, fill_value=False))]
    n_coinc = len(x)
    x = x[x.R <= 2]
    n_rad = len(x)
    g = x.groupby("id").agg(n_all=("id", "size"),
                            n_big=("E", lambda s: int((s >= 1e34).sum())),
                            maxE=("E", "max"), Teff=("Teff", "median")).reset_index()
    n_star = len(g)
    g["id"] = g.id.astype("int64")
    g = g.merge(ts[ts.pref == pref][["id", "Sum"]], on="id", how="inner")
    g["pref"] = pref
    stages[name] = {"raw": n_raw, "dedupe": n_dedup, "no_coincident": n_coinc,
                    "Rad<=2": n_rad, "stars_left": n_star, "with_obs_time": len(g)}
    return g

S = pd.concat([stars_from("TESS", "TIC", "TIC"), stars_from("Kepler", "KIC", "KIC"),
               stars_from("K2", "EPIC", "EPIC")], ignore_index=True)
for k, s in stages.items(): print(k, s)

# planets
ps  = pd.read_csv(rf"{D}\PSCompPars_2026.10.16_13.03.27.csv", comment="#", low_memory=False)
toi = pd.read_csv(rf"{D}\TOI_2026.10.16_13.07.57.csv", comment="#", low_memory=False)
k2  = pd.read_csv(rf"{D}\k2pandc_2026.10.16_13.09.00.csv", comment="#", low_memory=False)
koi = pd.read_csv(rf"{D}\koi_cumulative.csv.csv", comment="#", low_memory=False)
if "default_flag" in k2.columns: k2 = k2[k2.default_flag == 1]
koi = koi[col(koi, "koi_disposition").astype(str).str.upper() == "CONFIRMED"]
print("confirmed KOIs:", len(koi))

kn = col(koi, "kepler_name").where(col(koi, "kepler_name").notna(), col(koi, "kepoi_name"))
pk = pd.DataFrame({"pref": "KIC", "id": pd.to_numeric(koi.kepid, errors="coerce"), "tic": np.nan,
                   "planet": kn.astype(str), "a_cat": col(koi, "koi_sma"), "per": col(koi, "koi_period"),
                   "mass": 10 ** (col(koi, "koi_slogg") - 4.438) * col(koi, "koi_srad") ** 2,
                   "basis": "kepler3_logg_radius", "prio": 0})
tic_k2 = num(k2.tic_id)
p2 = pd.DataFrame({"pref": "EPIC", "id": num(k2.epic_hostname), "tic": tic_k2,
                   "planet": col(k2, "pl_name").astype(str), "a_cat": col(k2, "pl_orbsmax"),
                   "per": col(k2, "pl_orbper"), "mass": col(k2, "st_mass"),
                   "basis": "kepler3_period_mass", "prio": 1})
pt = pd.concat([
    pd.DataFrame({"pref": "TIC", "id": num(ps.tic_id), "tic": np.nan, "planet": col(ps, "pl_name").astype(str),
                  "a_cat": col(ps, "pl_orbsmax"), "per": col(ps, "pl_orbper"), "mass": col(ps, "st_mass"),
                  "basis": "kepler3_period_mass", "prio": 2}),
    pd.DataFrame({"pref": "TIC", "id": tic_k2, "tic": np.nan, "planet": col(k2, "pl_name").astype(str),
                  "a_cat": col(k2, "pl_orbsmax"), "per": col(k2, "pl_orbper"), "mass": col(k2, "st_mass"),
                  "basis": "kepler3_period_mass", "prio": 3}),
    pd.DataFrame({"pref": "TIC", "id": num(toi.tid), "tic": np.nan,
                  "planet": "TOI-" + pd.to_numeric(col(toi, "toi"), errors="coerce").map("{:.2f}".format),
                  "a_cat": np.nan, "per": col(toi, "pl_orbper"),
                  "mass": 10 ** (col(toi, "st_logg") - 4.438) * col(toi, "st_rad") ** 2,
                  "basis": "kepler3_logg_radius", "prio": 4})], ignore_index=True)

PL = pd.concat([pk, p2, pt], ignore_index=True)
PL["id"] = pd.to_numeric(PL.id, errors="coerce")
PL = PL.dropna(subset=["id"]).copy()
PL["id"] = PL.id.astype("int64")
PL = PL.merge(S[["pref", "id"]].drop_duplicates(), on=["pref", "id"])
PL = PL.sort_values(["pref", "id", "prio"])
keep = []
for _, g in PL.groupby(["pref", "id"]):
    kept = []
    for i, r in g.iterrows():
        dup = pd.notna(r.per) and any(pd.notna(k) and k > 0 and abs(r.per - k) / k < 0.02 for k in kept)
        if not dup: keep.append(i)
        if pd.notna(r.per): kept.append(r.per)
PL = PL.loc[keep].copy()
a_calc = (PL.mass * (PL.per / 365.25) ** 2) ** (1 / 3)
PL["koi_sma"] = PL.a_cat.fillna(a_calc)
PL["dist_basis"] = np.where(PL.a_cat.notna(), "catalogued", PL.basis)
PL["star_id"] = PL.pref + PL.id.astype(str)
m = (PL.pref == "EPIC") & PL.tic.notna()
PL.loc[m, "star_id"] = "TIC" + PL.loc[m, "tic"].astype("int64").astype(str)
n_pl = len(PL)
PL = PL[PL.koi_sma.notna() & (PL.koi_sma <= 10)]
PL = PL[~PL.star_id.isin(have_ids) & ~PL.planet.str.strip().isin(have_names)]

d = PL.merge(S, on=["pref", "id"])
d["obs_months"] = d.Sum / 30.4375
d["flare_rate_month"] = d.n_big / d.obs_months
d["flare_rate_all"] = d.n_all / d.obs_months
d["max_logE"] = np.log10(d.maxE)
ref = v[v.Teff.notna() & v.star_type.notna()]
knn = KNeighborsClassifier(15).fit(ref[["Teff"]], ref.star_type.astype(int))
d = d[d.Teff.notna()].copy()
d["star_type"] = knn.predict(d[["Teff"]])
d["source"] = "Su2022_" + d.pref.map({"TIC": "TESS", "KIC": "Kepler", "EPIC": "K2"})
d = d.sort_values("obs_months", ascending=False).drop_duplicates(["star_id", "planet"]).reset_index(drop=True)
d["R_eff"] = d.flare_rate_month * 0.47 * (0.16 / d.koi_sma) ** 2
d["label"] = np.where(d.R_eff >= 1, "High", np.where(d.R_eff >= 0.1, "Medium", "Low"))
d["baseline_label"] = np.where(d.flare_rate_month >= 1, "High", np.where(d.flare_rate_month >= 0.1, "Medium", "Low"))
d["rate_basis"] = "count"; d["obs_basis"] = "Su2022_Table8"; d["obs_assumed"] = False
d["no_detection"] = False; d["detected_big"] = d.n_big > 0
d["rate_capped"] = False; d["flare_rate_month_raw"] = d.flare_rate_month

print("planet rows after merge/dedupe:", n_pl, "| NEW rows:", len(d), "| stars:", d.star_id.nunique())
print(d.groupby("source").agg(rows=("planet", "size"), stars=("star_id", "nunique")).to_string())
print(pd.crosstab(d.source, d.label).to_string())
print("dist_basis:", d.dist_basis.value_counts().to_dict())
print("obs_months:", d.obs_months.describe().round(2).to_dict())
print("rate above 11.45:", int((d.flare_rate_month > 11.45).sum()))
print(d.nlargest(8, "flare_rate_month")[["source", "planet", "n_all", "n_big", "obs_months",
      "flare_rate_month", "koi_sma", "label"]].to_string())

d.to_csv(rf"{D}\su2022_new_rows.csv", index=False)
cols = [c for c in v.columns if c != "split"]
all11 = pd.concat([v[cols], d.reindex(columns=cols)], ignore_index=True)
print("TOTAL rows:", len(all11), "| stars:", all11.star_id.nunique())
print(all11.label.value_counts().to_dict())
all11.to_csv(rf"{D}\training_v11.csv", index=False)

In [ ]:
import pandas as pd
D = r"D:\Research\data"
x = pd.read_csv(rf"{D}\a103_J_A+A_689_A103_tablea1.csv")
print("A103", x.shape, list(x.columns))
with open(rf"{D}\ajad7956t9_mrt.txt", errors="replace") as f:
    for i, l in enumerate(f):
        if i > 45: break
        print(l.rstrip()[:110])

In [ ]:
import os
for r, d, f in os.walk(r"D:\Research\data\shi2025"):
    for x in f:
        print(os.path.join(r, x).replace(r"D:\Research\data", ""), os.path.getsize(os.path.join(r, x)) // 1024, "KB")

In [ ]:
import os, zipfile
dst = r"D:\Research\data\shi2025"
os.makedirs(dst, exist_ok=True)
dl = os.path.expanduser(r"~\Downloads")
for name in ["stellar.zip", "code.zip"]:
    p = os.path.join(dl, name)
    if os.path.exists(p):
        zipfile.ZipFile(p).extractall(os.path.join(dst, name[:-4]))
        print("extracted", name)
    else:
        print("not found in Downloads:", name)
for r, d, f in os.walk(dst):
    for x in f:
        print(os.path.relpath(os.path.join(r, x), dst), os.path.getsize(os.path.join(r, x)) // 1024, "KB")

In [ ]:
import pandas as pd, os, re
B = r"D:\Research\data\shi2025\stellar\stellar"
for f in ["catalog.csv", "features.csv", "flares_energy.csv", "stellar_params_with_type.csv"]:
    x = pd.read_csv(os.path.join(B, f))
    print("\n==", f, x.shape)
    print(list(x.columns)[:16])
    print(x.head(2).to_string()[:500])
print("\n== predict.py")
print(open(r"D:\Research\data\shi2025\code\code\predict.py", errors="replace").read())

cat = pd.read_csv(os.path.join(B, "catalog.csv"))
tc = next((c for c in cat.columns if "tic" in c.lower() or "star" in c.lower() or "name" in c.lower()), None)
print("\nstar-ID column guess:", tc)
if tc:
    ids = set(pd.to_numeric(cat[tc].astype(str).str.extract(r"(\d+)")[0], errors="coerce").dropna().astype(int))
    v = pd.read_csv(r"D:\Research\data\training_v11_split.csv")
    mine = set(pd.to_numeric(v.star_id[v.star_id.astype(str).str.startswith("TIC")]
                             .str.extract(r"(\d+)")[0], errors="coerce").dropna().astype(int))
    print("stars in catalogue:", len(ids), "| my TIC stars:", len(mine),
          "| covered:", len(ids & mine), "| my total stars:", v.star_id.nunique())

In [ ]:
import pandas as pd, os
B = r"D:\Research\data\shi2025\stellar\stellar"
print(open(r"D:\Research\data\shi2025\code\code\predict.py", errors="replace").read())

cat = pd.read_csv(os.path.join(B, "catalog.csv"))
cat["tic"] = pd.to_numeric(cat.Star_Name.astype(str).str.extract(r"(\d+)")[0], errors="coerce")
print("\noverall CME flag share:", round(cat.CME_Association.mean(), 3), "| flares:", len(cat))
print("flag share, Energy >= 1e34:", round(cat[cat.Energy >= 1e34].CME_Association.mean(), 3))

v = pd.read_csv(r"D:\Research\data\training_v11_split.csv")
mine = set(pd.to_numeric(v.star_id[v.star_id.astype(str).str.startswith("TIC")]
                         .str.extract(r"(\d+)")[0], errors="coerce").dropna().astype(int))
m = cat[cat.tic.isin(mine)]
g = m.groupby("tic").CME_Association.agg(["size", "sum"])
print("\nmy stars in catalogue:", len(g), "| flares:", len(m))
print("flares per star:", g["size"].describe().round(1).to_dict())
print("stars with at least 1 flagged flare:", int((g["sum"] > 0).sum()))

In [ ]:
import pandas as pd, os
B = r"D:\Research\data\shi2025\stellar\stellar"
cat = pd.read_csv(os.path.join(B, "catalog.csv"))
cat["tic"] = pd.to_numeric(cat.Star_Name.astype(str).str.extract(r"(\d+)")[0], errors="coerce")
v = pd.read_csv(r"D:\Research\data\training_v11_split.csv")
mine = set(pd.to_numeric(v.star_id[v.star_id.astype(str).str.startswith("TIC")]
                         .str.extract(r"(\d+)")[0], errors="coerce").dropna().astype(int))
m = cat[cat.tic.isin(mine)]
print("catalogue share:", round(cat.CME_Association.mean(), 3))
print("my 299 flares share:", round(m.CME_Association.mean(), 3))
print("my flares >= 1e34:", len(m[m.Energy >= 1e34]), "share:", round(m[m.Energy >= 1e34].CME_Association.mean(), 3))